In [25]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/05 20:17:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-07-13-Jul"
TABLE_NAME = "videos"

execution_date = "2026-07-13-Jul"
full_file_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [3]:
dfv = spark.read.parquet(full_file_path).cache()
dfv.createOrReplaceTempView("v_table")

26/08/05 20:17:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [29]:
dfv.show(3,truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------
 _id          | 6a5517ece0cdca20cb313a68                                                                                                                    
 competition  | {Kings_Cup, Kings Cup, [Al_Khaleej, Al_Bukiryah]}                                                                                           
 content_type | live                                                                                                                                        
 created_at   | 2026-06-25 18:48:33.057                                                                                                                     
 description  | Both decade happen opportunity bag. Teach always relate effect cut. Commercial research best.                                               
 duration_sec | 4799                                      

In [ ]:
# Document Structer

In [5]:
dfv.printSchema()

root
 |-- _id: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- teams: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- content_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- last_updated: timestamp (nullable = true)
 |-- match_id: string (nullable = true)
 |-- stats: struct (nullable = true)
 |    |-- views: integer (nullable = true)
 |    |-- likes: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical: struct (nullable = true)
 |    |-- max_resolution: string (nullable = true)
 |    |-- codec: string (nullable = true)
 |-- title: s

In [ ]:
# Nested Objects

In [31]:
dfv.select(["competition"]).limit(2).show(truncate=False, vertical=True)

dfv.filter(
    (dfv["competition.id"].isNull())
).count()

dfv.filter(
    (dfv["competition.name"].isNull())
).count()

dfv.filter(
    (dfv["competition.teams"].isNull())
).count()

dfv.select(["stats"]).limit(2).show(truncate=False, vertical=True)

dfv.select(["technical"]).limit(2).show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------
 competition | {Kings_Cup, Kings Cup, [Al_Khaleej, Al_Bukiryah]} 
-RECORD 1--------------------------------------------------------
 competition | {Kings_Cup, Kings Cup, [Al_Najma, Al_Ain]}        

-RECORD 0----------------
 stats | {134383, 14137} 
-RECORD 1----------------
 stats | {482675, 29357} 

-RECORD 0-------------------
 technical | {1080p, H.265} 
-RECORD 1-------------------
 technical | {4k, H.264}    



In [10]:
dfv.filter(
    (dfv["competition.id"].isNull())
).count()

dfv.filter(
    (dfv["competition.name"].isNull())
).count()

dfv.filter(
    (dfv["competition.teams"].isNull())
).count()

0

In [33]:
# Arrays

In [34]:
dfv.select(["languages"]).limit(2).show(truncate=False, vertical=True)
dfv.select(["tags"]).limit(2).show(truncate=False, vertical=True)
dfv.select(["competition.teams"]).limit(2).show(truncate=False, vertical=True)

-RECORD 0-------------
 languages | [en, ar] 
-RECORD 1-------------
 languages | [ar, en] 

-RECORD 0-------------------------------
 tags | [VAR, Super_Cup, VAR]           
-RECORD 1-------------------------------
 tags | [RSL, highlights, live, replay] 

-RECORD 0--------------------------
 teams | [Al_Khaleej, Al_Bukiryah] 
-RECORD 1--------------------------
 teams | [Al_Najma, Al_Ain]        



In [ ]:
# Data Types

In [35]:
# Missing Values

In [10]:
dfv.select([
    F.count(F.when(F.col(c).isNull(),
c)).alias(c)
    for c in dfv.columns
]).show(vertical=True)

-RECORD 0-------------
 _id          | 0     
 competition  | 0     
 content_type | 0     
 created_at   | 0     
 description  | 3971  
 duration_sec | 16579 
 languages    | 0     
 last_updated | 0     
 match_id     | 2538  
 stats        | 0     
 status       | 0     
 tags         | 0     
 technical    | 0     
 title        | 0     
 user_id      | 0     
 video_id     | 0     



In [11]:
# Uniqueness

In [13]:
dfv.groupBy("video_id").count().filter("count > 1").show()

+--------+-----+
|video_id|count|
+--------+-----+
+--------+-----+



In [ ]:
dfv.groupBy("user_id").count().filter("count > 1").show()

In [14]:
dfv.groupBy("content_type").count().show()

+------------+-----+
|content_type|count|
+------------+-----+
|      replay|16810|
|        live|16586|
|   highlight|16604|
+------------+-----+



In [15]:
dfv.groupBy("languages").count().show()

+---------+-----+
|languages|count|
+---------+-----+
| [ar, en]|11959|
|     [ar]|12025|
|       []| 1973|
| [en, ar]|12069|
|     [en]|11974|
+---------+-----+



In [16]:
dfv.groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|    ended|16759|
| archived|16662|
|streaming|16579|
+---------+-----+



In [18]:
dfv.groupBy("technical.max_resolution").count().show()

+--------------+-----+
|max_resolution|count|
+--------------+-----+
|          720p|12164|
|          480p|12389|
|          NULL| 1036|
|         1080p|12205|
|            4k|12206|
+--------------+-----+



In [19]:
dfv.groupBy("technical.codec").count().show()

+-----+-----+
|codec|count|
+-----+-----+
|H.265|24842|
|H.264|25158|
+-----+-----+



In [ ]:
# Optionl Fields

In [ ]:
"""
Optionl Fields cannot be distinguished from
null values after loading into a unified parwuet shcema.
"""

In [ ]:
# Array Quality

In [4]:
dfv.filter(
    F.size(F.col("languages")) == 0
).count()

1973

In [5]:
dfv.filter(
    F.size(F.col("tags")) == 0
).count()

0

In [6]:
dfv.filter(
    F.size(F.col("competition.teams")) == 0
).count()

0

In [7]:
dfv.filter(
    F.size("tags") !=
    F.size(F.array_distinct("tags"))
).count()

4997

In [8]:
dfv.filter(
    F.size("languages") !=
    F.size(F.array_distinct("languages"))
).count()

0

In [9]:
dfv.filter(
    F.size("competition.teams") !=
    F.size(F.array_distinct("competition.teams"))
).count()

0

In [11]:
dfv.groupBy("competition.name").count().orderBy("count", ascending=False).show()

+---------+-----+
|     name|count|
+---------+-----+
|Super Cup|16739|
|Kings Cup|16700|
|      RSL|16561|
+---------+-----+



In [12]:
dfv.groupBy("content_type").count().orderBy("count", ascending=False).show()

+------------+-----+
|content_type|count|
+------------+-----+
|      replay|16810|
|   highlight|16604|
|        live|16586|
+------------+-----+



In [13]:
dfv.groupBy("technical.codec").count().orderBy("count", ascending=False).show()

+-----+-----+
|codec|count|
+-----+-----+
|H.264|25158|
|H.265|24842|
+-----+-----+



In [14]:
dfv.groupBy("technical.max_resolution").count().orderBy("count", ascending=False).show()

+--------------+-----+
|max_resolution|count|
+--------------+-----+
|          480p|12389|
|            4k|12206|
|         1080p|12205|
|          720p|12164|
|          NULL| 1036|
+--------------+-----+



In [ ]:
# Cardinality

In [15]:
dfv.select("user_id").distinct().count()

1000

In [16]:
dfv.select("competition.name").distinct().count()

3

In [ ]:
dfv.select("competition.name").distinct().count()

In [17]:
from pyspark.sql.functions import explode

In [18]:
dfv.select(explode("languages").alias(
"languages")) \
    .distinct() \
    .count()

2

In [ ]:
# Nested Consisency

In [19]:
dfv.filter(
    F.size("competition.teams") !=
2).count()

0

In [5]:
dfv.filter(
    (F.col("competition.id").isNull() &
    F.col("competition.name").isNotNull()) |
    (F.col("competition.id").isNull() &
    F.col("competition.name").isNotNull())
).count()

0

In [ ]:
# Business Logic & Consistency Checks for  Video Metadata

In [ ]:
# ============================================================
# A. content_type <-> status consistency
# ============================================================

In [5]:
# 1. live should be streaming
print("Live errors:", dfv.filter(
    (dfv["content_type"] == "live") &
    (dfv["status"] != "streaming")
).count())

# 2. replay should be archived
print("Replay:", dfv.filter(
    (dfv["content_type"] == "replay") &
    (dfv["status"] != "archived")
).count())

# 3. highlight should be ended
print("Kighlight:", dfv.filter(
    (dfv["content_type"] == "highlight") &
    (dfv["status"] != "ended")
).count())

Live errors: 11036
Replay: 11171
Kighlight: 11092


In [ ]:
# ============================================================
# B. content_type <-> duration_sec consistency
# ============================================================

In [6]:
# 1. live should have null duration
print("Live duration errors", dfv.filter(
    (dfv["content_type"] == "live") &
    (dfv["duration_sec"].isNotNull())
).count())

# 2. replay should have non-null duration
print("Replay duration errors", dfv.filter(
    (dfv["content_type"] == "replay") &
    (dfv["duration_sec"].isNull())
).count())

# 3. highlight should have non-null duration
print("Highligh duration errors", dfv.filter(
    (dfv["content_type"] == "highlight") &
    (dfv["duration_sec"].isNull())
).count())

Live duration errors 11036
Replay duration errors 5487
Highligh duration errors 5542


In [ ]:
# ============================================================
# C. duration_sec range validity
# ============================================================

In [8]:
# 1. negative or zero duration
print("negative or zero errors:", dfv.filter(
    (dfv["duration_sec"] <= 0)
).count())

# 2. unreasonably large duration (> 3 hours = 10800 sec)
print("unreasonably large errors:", dfv.filter(
    (dfv["duration_sec"] > 10800)
).count())

negative or zero errors: 1088
unreasonably large errors: 577


In [ ]:
# ============================================================
# D. competition.name <-> tags consistency
# ============================================================

In [10]:
# 1. RSL videos should have 'RSL' in tags
print("Missing RSL tag errors", dfv.filter(
    (dfv["competition.name"] == "RSL") &
    (~F.array_contains(dfv["tags"], "RSL"))
).count())

# 2. Kings Cup videos should have 'Kings_Cup' in tags
print("Missing Kings Cup tag errors", dfv.filter(
    (dfv["competition.name"] == "Kings Cup") &
    (~F.array_contains(dfv["tags"], "Kings_Cup"))
).count())

# 3. Super Cup videos should have 'Super_Cup' in tags
print("Missing Super Cup tag errors", dfv.filter(
    (dfv["competition.name"] == "Super Cup") &
    (~F.array_contains(dfv["tags"], "Super_Cup"))
).count())

Missing RSL tag errors 10964
Missing Kings Cup tag errors 11222
Missing Super Cup tag errors 11202


In [ ]:
# ============================================================
# E. competition.teams consistency
# ============================================================

In [11]:
# 1. teams array should have exactly 2 elements
print("Incorrect team count errors", dfv.filter(
    F.size("competition.teams") != 2
).count())

# 2. teams array should not have duplicates
print("Duplicate team errors", dfv.filter(
    F.size("competition.teams") !=
    F.size(F.array_distinct("competition.teams"))
).count())

Incorrect team count errors 0
Duplicate team errors 0


In [ ]:
# ============================================================
# F. match_id consistency
# ============================================================

In [12]:
# 1. match_id should start with 'match_' when not null
print("Invalid format errors:", dfv.filter(
    (dfv["match_id"].isNotNull()) &
    (~dfv["match_id"].startswith("match_"))
).count())

Invalid format errors: 0


In [ ]:
# ============================================================
# G. stats consistency
# ============================================================

In [13]:
# 1. likes should not exceed views
print("Likes exceeding views errors:", dfv.filter(
    (dfv["stats.likes"] > dfv["stats.views"])
).count())

# 2. if views is null, likes should be 0 or null
print("Likes with null views errors:", dfv.filter(
    (dfv["stats.views"].isNull()) &
    (dfv["stats.likes"] > 0)
).count())

Likes exceeding views errors: 2406
Likes with null views errors: 2593


In [ ]:
# ============================================================
# H. title consistency
# ============================================================

In [14]:
# 1. title should not be empty
print("Empty title errors:", dfv.filter(
    dfv["title"] == ""
).count())

Empty title errors: 1483


In [ ]:
# ============================================================
# I. technical consistency
# ============================================================

In [15]:
# 1. max_resolution should be one of known values
print("Invalid max resolution errors:", dfv.filter(
    (dfv["technical.max_resolution"].isNotNull()) &
    (~dfv["technical.max_resolution"].isin(["4k", "1080p", "720p", "480p"]))
).count())

# 2. codec should be one of known values
print("Invalid codec errors:", dfv.filter(
    (dfv["technical.codec"].isNotNull()) &
    (~dfv["technical.codec"].isin(["H.264", "H.265"]))
).count())

Invalid max resolution errors: 0
Invalid codec errors: 0


In [ ]:
# ============================================================
# J. languages consistency
# ============================================================

In [16]:
# 1. languages should only contain known values
dfv.select(F.explode("languages").alias("lang")) \
   .filter(~F.col("lang").isin(["ar", "en"])) \
   .distinct() \
   .show(truncate=False)

[Stage 68:>                                                         (0 + 2) / 2]

+----+
|lang|
+----+
+----+



In [ ]:
# ============================================================
# K. content_type <-> match_id consistency
# ============================================================

In [17]:
# 1. live and replay should have match_id
print("Missing id for live or replay errord:", dfv.filter(
    (dfv["content_type"].isin(["live", "replay"])) &
    (dfv["match_id"].isNull())
).count())

Missing id for live or replay errord: 1689


In [ ]:
# ============================================================
# L. status <-> duration_sec consistency
# ============================================================

In [18]:
# 1. streaming should have null duration
print("Streaming with duration errors:", dfv.filter(
    (dfv["status"] == "streaming") &
    (dfv["duration_sec"].isNotNull())
).count())

# 2. ended/archived should have non-null duration
print("Ended or archives missing duration:", dfv.filter(
    (dfv["status"].isin(["ended", "archived"])) &
    (dfv["duration_sec"].isNull())
).count())

Streaming with duration errors: 0
Ended or archives missing duration: 0


In [ ]:
# ============================================================
# M. competition.id <-> competition.name consistency
# ============================================================

In [19]:
# 1. id=RSL -> name=RSL
print("RSL id and name mismatch errors:", dfv.filter(
    (F.col("competition.id") == "RSL") &
    (F.col("competition.name") != "RSL")
).count())

# 2. id=Kings_Cup -> name=Kings Cup
print("Kings Cup id and name mismatch errors:", dfv.filter(
    (F.col("competition.id") == "Kings_Cup") &
    (F.col("competition.name") != "Kings Cup")
).count())

# 3. id=Super_Cup -> name=Super Cup
print("Super Cup id and name mismatch errors:", dfv.filter(
    (F.col("competition.id") == "Super_Cup") &
    (F.col("competition.name") != "Super Cup")
).count())

RSL id and name mismatch errors: 0
Kings Cup id and name mismatch errors: 0
Super Cup id and name mismatch errors: 0


In [ ]:
# ============================================================
# N. competition.id <-> competition.teams consistency
# ============================================================

In [20]:
RSL_TEAMS = [
    'Al_Hilal', 'Al_Nassr', 'Al_Ittihad', 'Al_Ahli', 'Al_Ettifaq', 'Al_Taawoun',
    'Al_Fateh', 'Al_Fayha', 'Al_Khaleej', 'Al_Okhdood', 'Al_Raed', 'Al_Riyadh',
    'Al_Shabab', 'Al_Wehda', 'Damac', 'Al_Qadsiah', 'Al_Orobah', 'Al_Kholood'
]
KINGS_CUP_TEAMS = RSL_TEAMS + [
    'Al_Jabalain', 'Al_Bukiryah', 'Al_Jandal', 'Al_Najma', 'Al_Ain', 'Al_Safa',
    'Al_Adalah', 'Al_Batin', 'Ohud', 'Jeddah', 'Al_Jubail', 'Hajer', 'Al_Shoalah', 'Al_Faisaly'
]
SUPER_CUP_TEAMS = ["Al_Hilal", "Al_Nassr", "Al_Ittihad", "Al_Ahli"]

In [21]:
# 1. Super Cup teams should only be from allowed list
dfv.filter(F.col("competition.id") == "Super_Cup") \
   .select(F.explode("competition.teams").alias("team")) \
   .filter(~F.col("team").isin(SUPER_CUP_TEAMS)) \
   .distinct() \
   .show(truncate=False)

+----+
|team|
+----+
+----+



[Stage 87:=============================>                            (1 + 1) / 2]

In [22]:
# 1. Super Cup teams should only be from allowed list
dfv.filter(F.col("competition.id") == "Kings_Cup") \
   .select(F.explode("competition.teams").alias("team")) \
   .filter(~F.col("team").isin(KINGS_CUP_TEAMS)) \
   .distinct() \
   .show(truncate=False)

+----+
|team|
+----+
+----+



In [23]:
# 1. Super Cup teams should only be from allowed list
dfv.filter(F.col("competition.id") == "RSL") \
   .select(F.explode("competition.teams").alias("team")) \
   .filter(~F.col("team").isin(RSL_TEAMS)) \
   .distinct() \
   .show(truncate=False)

+----+
|team|
+----+
+----+



In [ ]:
# ============================================================
# O. Summary: total rows with ANY business logic violation
# ============================================================

In [24]:
dfv.filter(
    # content_type <-> status
    ((dfv["content_type"] == "live") & (dfv["status"] != "streaming")) |
    ((dfv["content_type"] == "replay") & (dfv["status"] != "archived")) |
    ((dfv["content_type"] == "highlight") & (dfv["status"] != "ended")) |
    # content_type <-> duration
    ((dfv["content_type"] == "live") & dfv["duration_sec"].isNotNull()) |
    ((dfv["content_type"].isin(["replay", "highlight"])) & dfv["duration_sec"].isNull()) |
    # duration range
    (dfv["duration_sec"] <= 0) |
    # competition <-> tags
    ((dfv["competition.name"] == "RSL") & (~F.array_contains(dfv["tags"], "RSL"))) |
    ((dfv["competition.name"] == "Kings Cup") & (~F.array_contains(dfv["tags"], "Kings_Cup"))) |
    ((dfv["competition.name"] == "Super Cup") & (~F.array_contains(dfv["tags"], "Super_Cup"))) |
    # stats
    (dfv["stats.likes"] > dfv["stats.views"]) |
    ((dfv["stats.views"].isNull()) & (dfv["stats.likes"] > 0)) |
    # title
    (dfv["title"] == "") |
    # match_id
    ((dfv["match_id"].isNotNull()) & (~dfv["match_id"].startswith("match_")))
).count()

45227

In [4]:
dfv.filter(
    dfv["created_at"] > dfv["last_updated"]
).count()

0

In [5]:
dfv.filter(
    (F.col("competition.id").isNull() & F.col("competition.name").isNotNull()) |
    (F.col("competition.id").isNotNull() & F.col("competition.name").isNull())
).count()


0

In [6]:
dfv.filter(
    (dfv["stats.views"] == 0) &
    (dfv["stats.likes"] > 0)
).count()

0

In [7]:
dfv.filter(
    (dfv["status"] == "ended") &
    (dfv["duration_sec"].isNull())
).count()

0

In [8]:
dfv.filter(
    (dfv["status"] == "archived") &
    (dfv["duration_sec"].isNull())
).count()

0

In [9]:
dfv.filter(
    dfv["user_id"].isNull()
).count()

0

In [10]:
dfv.filter(
    dfv["video_id"].isNull()
).count()

0

In [11]:
KNOWN_TAGS = ['football', 'highlights', 'live', 'RSL', 'Kings_Cup', 'Super_Cup', 'goals', 'replay', 'VAR']
dfv.select(F.explode("tags").alias("tag")) \
   .filter(~F.col("tag").isin(KNOWN_TAGS)) \
   .distinct() \
   .show(truncate=False)

+---+
|tag|
+---+
+---+



[Stage 23:>                                                         (0 + 2) / 2]

In [12]:
dfv.filter(
    (dfv["content_type"] == "live") &
    (dfv["match_id"].isNull())
).count()

834

In [13]:
dfv.filter(
    (dfv["stats.views"] == 0) &
    (dfv["stats.likes"] > 0)
).count()

0

In [14]:
dfv.filter(
    (dfv["content_type"] == "highlight") &
    ((dfv["duration_sec"] < 10) | (dfv["duration_sec"] > 1800))
).count()

8761

In [15]:
dfv.filter(
    (dfv["content_type"] == "replay") &
    (dfv["duration_sec"] < 300)
).count()

378

In [16]:
dfv.filter(
    (dfv["status"] == "ended") &
    (dfv["duration_sec"].isNull())
).count()

0

In [17]:
dfv.filter(
    (dfv["status"] == "archived") &
    (dfv["duration_sec"].isNull())
).count()

0

In [ ]:
# Edge Cases